## AI Fairness with Microsoft Fairlearn

In this example notebook we use Microsoft Fairlearn packages to explore how to mitigate bias when approving loan applications. We will look specifically at gender when approving loans and try out post processing techniques.

Some questions to think about in this example are:
 1. Is our model treating different groups fairly?
 2. What are the trade-offs between accuracy and fairness?
 3. How can we improve fairness while maintaining business value?

In [ ]:
import time
import threading
from IPython.display import clear_output

def workshop_keepalive():
    """Keep session alive during workshop presentations"""
    count = 0
    while count < 200:  # Run for ~16 hours max
        time.sleep(300)  # 5 minutes
        count += 1
        clear_output(wait=True)
        print(f" Workshop session active - {time.strftime('%H:%M:%S')}")
        print(f" Runtime: {count * 5} minutes")
        print("Continue with the workshop content below...")

threading.Thread(target=workshop_keepalive, daemon=True).start()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Fairlearn imports
from fairlearn.metrics import MetricFrame, selection_rate, demographic_parity_difference
from fairlearn.reductions import ExponentiatedGradient, DemographicParity, ErrorRate
from fairlearn.postprocessing import ThresholdOptimizer

In [ ]:
data = pd.read_csv('data/loan_data.csv')

In [ ]:
data

In [ ]:
print("Sample data:")
print(f"\nDataset size: {len(data)} applications")
print(f"Overall approval rate: {data['loan_approved'].mean():.2%}")
print(f"Approval rate by approval:")
print(data.groupby('gender')['loan_approved'].mean())
print(f"Average credit score by credit score:")
print(data.groupby('gender')['credit_score'].mean())
print(f"Average credit score by income:")
print(data.groupby('gender')['income'].mean())
print(f"Average credit score by employment:")
print(data.groupby('gender')['employment_years'].mean())
print(f"Average credit score by age:")
print(data.groupby('gender')['age'].mean())

### Step 1: Train a Standard Model

In [ ]:
# Prepare features (excluding the protected attribute 'gender')
# Prepare data for modeling
features = ['age', 'income', 'credit_score', 'employment_years','income_consistency_index']
X = data[['age', 'income', 'credit_score', 'employment_years','income_consistency_index']]
y = data['loan_approved']
sensitive_features = data['gender']

In [ ]:
# train/test split
X_train, X_test, y_train, y_test, A_train, A_test = train_test_split(
    X, y, sensitive_features, test_size=0.3, random_state=42
)

In [ ]:
# Train a standard model
standard_model = DecisionTreeClassifier(
    max_depth=4,           
    min_samples_split=20,
    random_state=42
)
standard_model.fit(X_train, y_train)

# Make predictions
y_pred_standard = standard_model.predict(X_test)
accuracy_standard = accuracy_score(y_test, y_pred_standard)

print(f"Standard Model Accuracy: {accuracy_standard:.3f}")

### Step 2: Assess Fairness with Fairlearn Metrics

In [ ]:
# Create MetricFrame to analyze performance across groups
metric_frame = MetricFrame(
    metrics={
        'accuracy': accuracy_score,
        'selection_rate': selection_rate  # Approval rate
    },
    y_true=y_test,
    y_pred=y_pred_standard,
    sensitive_features=A_test
)

print("Performance by Gender Group:")
print(metric_frame.by_group)
print("\nOverall Performance:")
print(metric_frame.overall)

In [ ]:
# Calculate fairness metrics
dp_diff = demographic_parity_difference(
    y_test, y_pred_standard, sensitive_features=A_test
)

print(f"Demographic Parity Difference: {dp_diff:.3f}")
print("(Values closer to 0 indicate more fair treatment)")

### Step 3: Fairness Visualization

In [ ]:
# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: Approval rates by gender
approval_rates = metric_frame.by_group['selection_rate']
approval_rates.plot(kind='bar', ax=axes[0], color=['lightcoral', 'lightblue'])
axes[0].set_title('Loan Approval Rates by Gender')
axes[0].set_ylabel('Approval Rate')
axes[0].set_xlabel('Gender')
axes[0].tick_params(axis='x', rotation=0)

# Plot 2: Accuracy by gender
accuracy_by_group = metric_frame.by_group['accuracy']
accuracy_by_group.plot(kind='bar', ax=axes[1], color=['lightcoral', 'lightblue'])
axes[1].set_title('Model Accuracy by Gender')
axes[1].set_ylabel('Accuracy')
axes[1].set_xlabel('Gender')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

# Summary
print("\n" + "="*50)
print("FAIRNESS ASSESSMENT")
print("="*50)
print(f"• Overall model accuracy: {accuracy_standard:.1%}")
print(f"• Male approval rate: {approval_rates['Male']:.1%}")
print(f"• Female approval rate: {approval_rates['Female']:.1%}")
print(f"• Approval rate gap: {abs(approval_rates['Male'] - approval_rates['Female']):.1%}")
print("• Recommendation: Model shows potential gender bias")

### Step 4: Apply Fairlearn Packages

In [ ]:
# Post-processing with Threshold Optimization
threshold_optimizer = ThresholdOptimizer(
    estimator=standard_model,
    constraints='demographic_parity',
    predict_method='predict_proba',
    prefit=True
)

threshold_optimizer.fit(X_train, y_train, sensitive_features=A_train)

# Make fair predictions
y_pred_fair = threshold_optimizer.predict(X_test, sensitive_features=A_test)

# Assess metrics
fair_metric_frame = MetricFrame(
    metrics={
        'accuracy': accuracy_score,
        'selection_rate': selection_rate
    },
    y_true=y_test,
    y_pred=y_pred_fair,
    sensitive_features=A_test
)

# Calculate new fairness metrics
dp_diff_fair = demographic_parity_difference(
    y_test, y_pred_fair, sensitive_features=A_test
)

print("FAIR MODEL RESULTS:")
print("Performance by Gender Group:")
print(fair_metric_frame.by_group)
print(f"\nNew Demographic Parity Difference: {dp_diff_fair:.3f}")
print(f"Improvement in fairness: {abs(dp_diff) - abs(dp_diff_fair):.3f}")

### Step 4: Compare models

In [ ]:
# Create comparison visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Comparison of approval rates
comparison_data = pd.DataFrame({
    'Standard Model': metric_frame.by_group['selection_rate'],
    'Fair Model': fair_metric_frame.by_group['selection_rate']
})

comparison_data.plot(kind='bar', ax=axes[0])
axes[0].set_title('Approval Rates: Standard vs Fair Model')
axes[0].set_ylabel('Approval Rate')
axes[0].set_xlabel('Gender')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=0)

# Accuracy comparison
accuracy_comparison = pd.DataFrame({
    'Standard Model': metric_frame.by_group['accuracy'],
    'Fair Model': fair_metric_frame.by_group['accuracy']
})

accuracy_comparison.plot(kind='bar', ax=axes[1])
axes[1].set_title('Accuracy: Standard vs Fair Model')
axes[1].set_ylabel('Accuracy')
axes[1].set_xlabel('Gender')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

# Executive summary of changes
print("\n" + "="*60)
print("BUSINESS IMPACT ANALYSIS")
print("="*60)
print(f"Standard Model:")
print(f"  • Overall Accuracy: {accuracy_standard:.4f}")
print(f"  • Fairness Gap: {abs(dp_diff):.4f}")

accuracy_fair = accuracy_score(y_test, y_pred_fair)
print(f"\nFair Model:")
print(f"  • Overall Accuracy: {accuracy_fair:.4f}")
print(f"  • Fairness Gap: {abs(dp_diff_fair):.4f}")

print(f"\nTrade-offs:")
print(f"  • Accuracy change: {accuracy_fair - accuracy_standard:+.4f}")
print(f"  • Fairness improvement: {abs(dp_diff) - abs(dp_diff_fair):+.4f}")

### Step 5: Explanation Dashboard

In [ ]:
from raiwidgets import ExplanationDashboard
from interpret.ext.blackbox import TabularExplainer

In [ ]:
class_labels = ['Not Approved', 'Approved']

In [ ]:
# 1. Using SHAP TabularExplainer
explainer = TabularExplainer(standard_model, 
                             X_train, 
                             features=features, 
                             classes=class_labels)

In [ ]:
global_explanation = explainer.explain_global(X_test)

In [ ]:
# Corresponding feature names
print('Ranked global importance names: {}'.format(global_explanation.get_ranked_global_names()))

# Per class feature names
print('Ranked per class feature names: {}'.format(global_explanation.get_ranked_per_class_names()))

### Section 6: Fairness Dashboard

Fairlearn has an interactive dashboard where you can see visualizations of the models, data points, etc.

In [ ]:
from raiwidgets import ResponsibleAIDashboard
from responsibleai import RAIInsights

In [ ]:
train_data = X_train.copy()
train_data['loan_approved'] = y_train

test_data = X_test.copy()
test_data['loan_approved'] = y_test

In [ ]:
target_feature = 'loan_approved'
rai_insights = RAIInsights(standard_model, train_data, test_data, target_feature, 'classification')

In [ ]:
# Add the components of the toolbox
# Interpretability
rai_insights.explainer.add()

In [ ]:
rai_insights.compute()

In [ ]:
try:
    # Get explanations
    explanations = rai_insights.explainer.get()
    if explanations:
        print(f"✓ Explanations available: {len(explanations)} explanation(s)")
        global_explanation = explanations[0]
        
        print("\nTop feature importances:")
        feature_importance = global_explanation.global_importance_values
        feature_names = global_explanation.features
        
        # Sort features by importance
        importance_pairs = list(zip(feature_names, feature_importance))
        importance_pairs.sort(key=lambda x: abs(x[1]), reverse=True)
        
        for feature, importance in importance_pairs:
            print(f"  {feature}: {importance:.4f}")
        
except Exception as e:
    print(f"Error accessing insights: {e}")

In [ ]:
try:
    from raiwidgets import ExplanationDashboard
    
    # Get the explanation
    explanation = rai_insights.explainer.get()[0]
    
    # Try the lightweight dashboard
    exp_dashboard = ExplanationDashboard(
        explanation, 
        standard_model, 
        dataset=X_test, 
        true_y=y_test,
        features=features
    )
    
    print("Explanation dashboard started - check for any output above")
    
except Exception as e:
    print(f"Error with explanation dashboard: {e}")